# w2 · Neural models (KKBox churn)

*Three listening/transaction/combined LSTMs (M5, M7, M8) and one hybrid MLP + listening
LSTM (M10), per-family Optuna search, one shared split, common metrics, confusion
matrices, final refit and reloadable scoring.*

The KPI is **log-loss** (probability quality). The real churn rate (~9%) is kept:
no SMOTE, no subsampling of classes, no manual rescaling of probabilities.

## Catalogue

| ID | Visible name | Input | Optimisation |
|----|--------------|-------|--------------|
| M5 | Listening LSTM | 90 × 8 | own Optuna |
| M7 | Transaction LSTM | 90 × 12 | own Optuna |
| M8 | Combined LSTM | 90 × 20 | own Optuna |
| M10 | Hybrid MLP + listening LSTM | Table + 90 × 8 | Optuna recurrent correction |

M9 (tabular MLP) is trained **only as M10's frozen backbone**; it is not a compared model.

## Month-over-month validation, holdout kept out of every decision

Same protocol as the w1 tabular notebook, so both notebooks are comparable:

| Part | Cohort | Features / sequence window ends | Label month | Use |
|---|---|---|---|---|
| `fit` / `stop` / `tune` | `train_feb` | 2017-01-31 | February | train, early stop, Optuna + threshold |
| `holdout` | `train` (March) | 2017-02-28 | March | single final evaluation |
| final refit | February + March | — | — | models used for April `test` |

- **development** = the whole February cohort, split once by a stable msno hash, stratified
  by label, into `fit` (~75%), `stop` (~12.5%) and `tune` (~12.5%).
- **holdout** = the whole March cohort. Its 90-day windows and timing features are built
  exactly like February's, one month later, so the tune → holdout gap measures month-to-month
  drift. Most users appear in both months with different snapshots and labels, as in
  production.

Optuna trains on `fit`, early-stops on `stop`, and reports log-loss on `tune`. Thresholds and
the winners are fixed on February **before** March is touched. Then each family is refit on all
of February and evaluated once on March, and finally refit on February + March for April
inference. Test has no labels. Churn prevalence differs between months (≈ 6.4 % in February,
≈ 9.0 % in March).

Inputs are split across two exports: the tabular tables come from the w1 tabular export and
the sequence tensors from the w2 sequence export. The non-neural models live in the w1
tabular notebook.


## 1. Configuration

`SMOKE_TEST=True` runs a tiny sample with few trials/epochs to validate the code; its output
goes to a separate folder and is clearly marked as a test. The default (`SMOKE_TEST=False`)
uses every eligible user and requires a GPU. Smoke results are never published as FULL.


In [1]:
from pathlib import Path
import gc
import json
import hashlib
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 110

# --- Run mode -------------------------------------------------------------
# A smoke test exercises every code path with a small sample, few trials and few
# epochs. Keep it True to validate quickly; set it to False for the full run.
SMOKE_TEST = False

# At most 50 Optuna trials TOTAL across families, not 50 per model.
# This is a search budget, not a guarantee of the global optimum.
# Refit training is separate from this budget.
OPTUNA_MAX_TRIALS = 50
TRIALS_FULL = {"M5": 5, "M7": 5, "M8": 5, "M9": 5, "M10": 5}
assert sum(TRIALS_FULL.values()) <= OPTUNA_MAX_TRIALS
TRIALS_SMOKE = {k: 2 for k in TRIALS_FULL}
TRIALS = TRIALS_SMOKE if SMOKE_TEST else TRIALS_FULL
TRAIN_VERBOSE = 1  # Keras batch/epoch bars.

# Deep-learning budgets.
DL_MAX_EPOCHS = 2 if SMOKE_TEST else 40
DL_PATIENCE = 2 if SMOKE_TEST else 5
# Smoke sample size (users) taken from each phase; None = use all users.
SMOKE_USERS = 10 if SMOKE_TEST else None
# Smoke also limits test rows scored at the end (None = all test users).
SMOKE_TEST_ROWS = 10 if SMOKE_TEST else None

N_DAYS = 90  # All sequences keep 90 calendar days in this delivery.

# Batch size for inference.
INFER_BATCH = 8192

print("SMOKE_TEST:", SMOKE_TEST, "| trials:", TRIALS,
      "| total trials:", sum(TRIALS.values()), "| DL epochs:", DL_MAX_EPOCHS)


SMOKE_TEST: False | trials: {'M5': 5, 'M7': 5, 'M8': 5, 'M9': 5, 'M10': 5} | total trials: 25 | DL epochs: 40


In [2]:
# --- Paths ----------------------------------------------------------------
# Tabular tables come from the w1 v4 tabular export; sequences from the w2 v4 sequence export.
# On Kaggle both are uploaded as one dataset.
ON_KAGGLE = Path("/kaggle/input").exists()
INPUT_DIR = Path("/kaggle/input/dataset-final") if ON_KAGGLE else Path("export-tabular-v4")
SEQ_DIR = Path("/kaggle/input/dataset-final") if ON_KAGGLE else Path("export-sequential-v4")
_base_out = Path("/kaggle/working") if ON_KAGGLE else Path("artifacts")
OUT_DIR = _base_out / ("sherlock_w2_smoke" if SMOKE_TEST else "sherlock_w2")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("INPUT_DIR:", INPUT_DIR, "| SEQ_DIR:", SEQ_DIR, "| OUT_DIR:", OUT_DIR, "| ON_KAGGLE:", ON_KAGGLE)


INPUT_DIR: export-tabular-v4 | SEQ_DIR: export-sequential-v4 | OUT_DIR: artifacts/sherlock_w2 | ON_KAGGLE: False


### Local path override (optional)

**Kaggle:** leave the next cell commented; the configuration above is correct.

**Local:** uncomment the lines to read `export-tabular-v4/` and `export-sequential-v4/` from the project folder.
These paths replace the ones above. Comment them again to return to Kaggle.

In [3]:
# LOCAL DATA: uncomment to use the local exports.
# ON_KAGGLE = False
# INPUT_DIR = Path("export-tabular-v4")
# SEQ_DIR = Path("export-sequential-v4")
# OUT_DIR = (Path("artifacts") / ("sherlock_w2_smoke" if SMOKE_TEST else "sherlock_w2"))
# OUT_DIR.mkdir(parents=True, exist_ok=True)
# print("LOCAL override ->", INPUT_DIR, SEQ_DIR, OUT_DIR)


In [ ]:
import os
import sys
import shutil
import subprocess
import tensorflow as tf

print("Python:", sys.executable)
print("nvidia-smi:", shutil.which("nvidia-smi"))
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("LD_LIBRARY_PATH:", os.environ.get("LD_LIBRARY_PATH"))
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as exc:
    print("nvidia-smi desde el kernel:", repr(exc))

2026-10-15 04:50:05.188370: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
# --- Library availability and GPU detection -------------------------------
# Import heavy libraries here so a missing dependency fails early and clearly.
from sklearn.metrics import (
    log_loss, roc_auc_score, average_precision_score, brier_score_loss,
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix, roc_curve,
    precision_recall_curve, classification_report,
)
from sklearn.calibration import calibration_curve

import optuna
import tensorflow as tf

optuna.logging.set_verbosity(optuna.logging.WARNING)

import random
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

GPUS = tf.config.list_physical_devices("GPU")
for gpu in GPUS:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as exc:
        print("GPU memory growth not set:", exc)
USE_GPU = len(GPUS) > 0
if not USE_GPU and not SMOKE_TEST:
    raise RuntimeError(
        "TensorFlow no detecta ninguna GPU; el entrenamiento completo de las LSTM "
        "sería demasiado lento. En Linux, instala tensorflow[and-cuda] en el "
        "entorno del kernel y reinícialo. Verifica con "
        "tf.config.list_physical_devices('GPU') antes de ejecutar Sherlock."
    )

# Keep float32 for stable log-loss; cuDNN LSTM needs no recurrent_dropout.
tf.keras.mixed_precision.set_global_policy("float32")

ENV = {
    "numpy": np.__version__, "pandas": pd.__version__,
    "sklearn": __import__("sklearn").__version__,
    "optuna": optuna.__version__, "tensorflow": tf.__version__,
    "gpus": len(GPUS),
}
print("Environment:", json.dumps(ENV, indent=2))


## 2. Dual v4 contract and loading

Validate `dataset_version == 4` for both the tabular export (feature catalogue) and the
sequence export (channel tensors), with the three cutoffs. Read channel names and order from
`metadata["sequences"]`, and confirm the two exports publish the same `msno` row hashes for
every cohort. The two labelled cohorts are stacked into one frame (`cohort` column: February
rows first, then March) and their tensors are exposed through one row-stacked view, so a row
index means the same user-month in the table and in the tensors. Verify shapes, dtype, binary
flags, and that the first 8 channels of the combined tensor match the listening tensor.


In [ ]:
TAB_META = json.loads((INPUT_DIR / "metadata.json").read_text())
SEQ_META = json.loads((SEQ_DIR / "metadata.json").read_text())
if TAB_META.get("dataset_version") != 4 or SEQ_META.get("dataset_version") != 4:
    raise ValueError("Expected dataset_version 4 for both the tabular and sequence exports.")
if set(SEQ_META.get("sequences", {})) != {"hybrid", "lstm"}:
    raise ValueError("Expected dual sequences {'hybrid','lstm'} in sequence metadata.")
EXPECTED_CUTOFFS = {"train_feb": "2017-01-31", "train": "2017-02-28", "test": "2017-03-31"}
if TAB_META["cutoffs"] != EXPECTED_CUTOFFS or SEQ_META["cutoffs"] != TAB_META["cutoffs"]:
    raise ValueError("Unexpected or mismatched cutoffs between exports.")
DEV_COHORT, HOLD_COHORT = "train_feb", "train"   # February trains, March is the holdout
for _split in EXPECTED_CUTOFFS:
    assert TAB_META[f"hash_{_split}"] == SEQ_META[f"hash_{_split}"], f"{_split} rows differ"

FEATURES = TAB_META["features"]
CATEGORICAL = TAB_META["categorical"]
LISTEN_CHANNELS = SEQ_META["sequences"]["hybrid"]["channels"]
LSTM_CHANNELS = SEQ_META["sequences"]["lstm"]["channels"]
TX_CHANNELS = LSTM_CHANNELS[8:]           # 12 transaction/state channels for M7
TX_SPECS = SEQ_META["sequences"]["lstm"]["channel_specs"]

assert LISTEN_CHANNELS[0] == "num_25" and LISTEN_CHANNELS[-1] == "active"
assert LSTM_CHANNELS[:8] == LISTEN_CHANNELS and len(LSTM_CHANNELS) == 20
assert len(TX_CHANNELS) == 12
assert not {"msno", "is_churn"}.intersection(FEATURES)

print("dataset_version:", SEQ_META["dataset_version"],
      "| features:", len(FEATURES), "| categorical:", CATEGORICAL)
print("listening channels:", LISTEN_CHANNELS)
print("transaction channels:", TX_CHANNELS)


In [ ]:
# Read the two labelled cohorts and test; stack the cohorts with an explicit row index.
def _order_hash(msno_series):
    digest = hashlib.sha1()
    for value in msno_series:
        digest.update(str(value).encode()); digest.update(b"\n")
    return digest.hexdigest()

cohort_frames = {}
for cohort in (DEV_COHORT, HOLD_COHORT):
    frame = pd.read_parquet(INPUT_DIR / f"{cohort}.parquet",
                            columns=["msno", "is_churn"] + FEATURES).reset_index(drop=True)
    assert frame["msno"].is_unique, f"{cohort} has duplicated msno"
    assert _order_hash(frame["msno"]) == TAB_META[f"hash_{cohort}"], f"{cohort} parquet misaligned"
    frame.insert(1, "cohort", cohort)
    cohort_frames[cohort] = frame
COHORT_SIZES = {c: len(f) for c, f in cohort_frames.items()}

train_df = pd.concat([cohort_frames[DEV_COHORT], cohort_frames[HOLD_COHORT]], ignore_index=True)
train_df["cohort"] = train_df["cohort"].astype("category")
train_df["row"] = np.arange(len(train_df))
COHORT = train_df["cohort"].to_numpy(dtype=str)
del cohort_frames; gc.collect()

test_df = pd.read_parquet(INPUT_DIR / "test.parquet", columns=["msno"] + FEATURES).reset_index(drop=True)
assert test_df["msno"].is_unique
assert _order_hash(test_df["msno"]) == TAB_META["hash_test"], "test parquet misaligned"
test_df["row"] = np.arange(len(test_df))

y_all = train_df["is_churn"].to_numpy(dtype=np.int32)
assert set(np.unique(y_all)) == {0, 1}
for cohort in (DEV_COHORT, HOLD_COHORT):
    mask = COHORT == cohort
    print(f"{cohort:9s} rows: {mask.sum():>9,} | label month: {TAB_META['label_months'][cohort]}"
          f" | churn rate: {y_all[mask].mean():.4f}")
print("stacked train:", train_df.shape, "| test:", test_df.shape)


In [ ]:
# Memory-mapped tensors; never materialise the whole array in float32.
def _open_tensor(family, split):
    spec = SEQ_META["sequences"][family]
    assert spec["order"] == "oldest_to_newest_calendar_day"
    assert [s["name"] for s in spec["channel_specs"]] == spec["channels"]
    assert spec[split]["row_hash"] == SEQ_META[f"hash_{split}"]
    arr = np.load(SEQ_DIR / spec[split]["file"], mmap_mode="r")
    assert list(arr.shape) == spec[split]["shape"]
    assert arr.dtype == np.float16
    return arr


class StackedRows:
    """Read-only row-stacked view over memory-mapped cohort tensors (no copy).

    Global row ``r`` maps to the cohort part that contains it, in stacking order, so the view
    is indexed exactly like ``train_df``. Supports ``view[rows]``, ``view[rows, ...]`` and
    slices of rows; each access reads only the requested rows from disk.
    """
    def __init__(self, parts):
        self.parts = list(parts)
        self.offsets = np.cumsum([0] + [len(p) for p in self.parts])
        self.shape = (int(self.offsets[-1]),) + tuple(self.parts[0].shape[1:])
        self.dtype = self.parts[0].dtype
        assert all(p.shape[1:] == self.parts[0].shape[1:] and p.dtype == self.dtype
                   for p in self.parts)

    def __len__(self):
        return self.shape[0]

    def __getitem__(self, key):
        rest = ()
        if isinstance(key, tuple):
            key, rest = key[0], key[1:]
        rows = np.arange(self.shape[0])[key] if isinstance(key, slice) else np.asarray(key)
        scalar = rows.ndim == 0
        rows = np.atleast_1d(rows).astype(np.int64)
        if len(rows) and (rows.min() < 0 or rows.max() >= self.shape[0]):
            raise IndexError("row index out of range")
        out = np.empty((len(rows),) + self.shape[1:], dtype=self.dtype)
        part = np.searchsorted(self.offsets, rows, side="right") - 1
        for p in np.unique(part):
            sel = part == p
            out[sel] = self.parts[p][rows[sel] - self.offsets[p]]
        out = out[(slice(None),) + rest] if rest else out
        return out[0] if scalar else out


SEQ = {}
for _family in ("hybrid", "lstm"):
    SEQ[(_family, "train")] = StackedRows([_open_tensor(_family, DEV_COHORT),
                                           _open_tensor(_family, HOLD_COHORT)])
    SEQ[(_family, "test")] = _open_tensor(_family, "test")
assert SEQ[("hybrid", "train")].shape == (len(train_df), N_DAYS, 8)
assert SEQ[("lstm", "train")].shape == (len(train_df), N_DAYS, 20)
assert SEQ[("hybrid", "test")].shape == (len(test_df), N_DAYS, 8)
assert SEQ[("lstm", "test")].shape == (len(test_df), N_DAYS, 20)

# Spot-check alignment at both ends of each cohort: hybrid == first 8 lstm channels, binary flags.
_n_dev = COHORT_SIZES[DEV_COHORT]
_probe_rows = np.r_[0:32, _n_dev - 32:_n_dev + 32, len(train_df) - 32:len(train_df)]
_probe = np.asarray(SEQ[("lstm", "train")][_probe_rows], dtype=np.float32)
np.testing.assert_array_equal(_probe[:, :, :8],
                              np.asarray(SEQ[("hybrid", "train")][_probe_rows], dtype=np.float32))
np.testing.assert_array_equal(SEQ[("lstm", "train")][_n_dev],
                              SEQ[("lstm", "train")].parts[1][0])
for ch in ["tx_state_known", "tx_last_is_cancel", "tx_last_is_auto_renew",
           "tx_expiry_known", "tx_state_ambiguous", "active"]:
    assert np.isin(_probe[:, :, LSTM_CHANNELS.index(ch)], [0, 1]).all(), ch
del _probe
print("tensors mapped (February + March stacked) and alignment spot-checked")


## 3. Single shared split (fit / stop / tune / holdout)

One protocol for every model, identical to w1. The **holdout** is the whole March cohort
(`train_v2` labels, sequences ending 2017-02-28). **Development** is the whole February cohort
(`train.csv` labels, sequences ending 2017-01-31), split once by a stable msno hash stratified
by label into `fit` (~75%), `stop` (~12.5%) and `tune` (~12.5%).

These are internal parts of a single evaluation, not alternative experiments. Every model uses
the same rows in each part. We validate disjointness, coverage, both classes and order, and
save sizes, prevalences and the `days_to_expire` distribution per part in one diagnostic.


In [ ]:
def _msno_hash_unit(msno):
    """Stable hash in [0, 1) from msno, independent of RNG call order."""
    h = hashlib.sha1(("salt110:" + str(msno)).encode()).hexdigest()
    return int(h[:15], 16) / float(16 ** 15)

hash_unit = train_df["msno"].map(_msno_hash_unit).to_numpy()

# Holdout: the whole March cohort. Development: the whole February cohort.
holdout_rows = np.flatnonzero(COHORT == HOLD_COHORT)
dev_rows = np.flatnonzero(COHORT == DEV_COHORT)

# Development split by stable hash, stratified by label (no repeated RNG draws).
# Thresholds on the per-class hash rank give exact, reproducible stratified shares.
def _stratified_hash_assign(rows, shares):
    """Assign each row to a part using its per-class hash rank. shares sum to 1."""
    assign = np.empty(len(rows), dtype=object)
    names = list(shares)
    cum = np.cumsum([shares[n] for n in names])
    for cls in (0, 1):
        cls_rows = rows[y_all[rows] == cls]
        ranks = np.argsort(np.argsort(hash_unit[cls_rows])) / max(len(cls_rows), 1)
        part_idx = np.searchsorted(cum, ranks, side="right")
        part_idx = np.clip(part_idx, 0, len(names) - 1)
        assign[np.searchsorted(rows, cls_rows)] = np.asarray(names, dtype=object)[part_idx]
    return assign

dev_shares = {"fit": 0.75, "stop": 0.125, "tune": 0.125}
dev_assign = _stratified_hash_assign(dev_rows, dev_shares)
SPLIT = {
    "fit": np.sort(dev_rows[dev_assign == "fit"]),
    "stop": np.sort(dev_rows[dev_assign == "stop"]),
    "tune": np.sort(dev_rows[dev_assign == "tune"]),
    "holdout": holdout_rows,
}
DEV_ROWS = dev_rows  # all development rows (fit+stop+tune), for the development refit

In [ ]:
# Optional smoke subsample: shrink every part but keep both classes and disjointness.
if SMOKE_USERS is not None:
    rng = np.random.default_rng(SEED)
    sized = {part: SMOKE_USERS for part in SPLIT}
    small = {}
    for part, idx in SPLIT.items():
        take = min(sized[part], len(idx))
        pos = idx[y_all[idx] == 1]
        neg = idx[y_all[idx] == 0]
        n_pos = max(1, int(round(take * len(pos) / len(idx))))
        n_pos = min(n_pos, len(pos)); n_neg = min(take - n_pos, len(neg))
        chosen = np.concatenate([rng.choice(pos, n_pos, replace=False),
                                 rng.choice(neg, n_neg, replace=False)])
        small[part] = np.sort(chosen)
    SPLIT = small
    DEV_ROWS = np.sort(np.concatenate([SPLIT["fit"], SPLIT["stop"], SPLIT["tune"]]))
    print("SMOKE split sizes:", {k: len(v) for k, v in SPLIT.items()})

In [ ]:
# Validate: disjoint, both classes, sorted; save indices and a single diagnostic.
_seen = {}
for part, idx in SPLIT.items():
    assert np.all(np.diff(idx) > 0), f"{part} not sorted/unique"
    assert set(np.unique(y_all[idx])) == {0, 1}, f"{part} missing a class"
    for other, oidx in _seen.items():
        assert not np.intersect1d(idx, oidx).size, f"{part} overlaps {other}"
    _seen[part] = idx
assert set(COHORT[SPLIT["holdout"]]) == {HOLD_COHORT}
assert all(set(COHORT[SPLIT[p]]) == {DEV_COHORT} for p in ("fit", "stop", "tune"))

_dte = train_df["days_to_expire"].to_numpy()
split_summary = pd.DataFrame([
    {"part": part, "cohort": COHORT[idx[0]],
     "label_month": TAB_META["label_months"][COHORT[idx[0]]], "n_users": len(idx),
     "churn_rate": round(float(y_all[idx].mean()), 5),
     "n_churn": int(y_all[idx].sum()), "n_stay": int((y_all[idx] == 0).sum()),
     "days_to_expire_p01": float(np.percentile(_dte[idx], 1)),
     "days_to_expire_p50": float(np.percentile(_dte[idx], 50)),
     "days_to_expire_p99": float(np.percentile(_dte[idx], 99))}
    for part, idx in SPLIT.items()
])
np.savez_compressed(OUT_DIR / "split_indices.npz",
                    **{k: v for k, v in SPLIT.items()}, development=DEV_ROWS)
split_summary.to_csv(OUT_DIR / "split_summary.csv", index=False)
FIT_RATE = float(y_all[SPLIT["fit"]].mean())   # training prevalence for M0 and biases
print("saved split_indices.npz and split_summary.csv | fit churn rate:", round(FIT_RATE, 5))
split_summary

## 4. Phase preprocessors and bounded-memory batching

Scalers, imputation and categories are fit **only** on each phase's training users
(`fit`, all of February for the development refit, or February + March for the final
refit).
We never union train+test categories. The first 8 channels of the combined and listening
tensors share exactly the same listening transform.

Tensors are read with mmap and consumed in index batches: we never call `np.asarray` on
the whole array in float32 nor materialise all four parts at once. Quantiles for the
listening and transaction scalers use a bounded deterministic sample of **fit** days
only (that sample is not a user subsample); the mean/std use all valid values in blocks.

In [ ]:
QUANTILE_SAMPLE = 2_000_000   # max day-rows used to estimate quantiles (fit only)

def _listening_block_iter(rows, block=20000):
    """Yield float32 blocks of the listening tensor for the given train rows."""
    arr = SEQ[("hybrid", "train")]
    for start in range(0, len(rows), block):
        idx = rows[start:start + block]
        yield np.asarray(arr[idx, -N_DAYS:, :], dtype=np.float32)


def fit_sequence_scaler(rows):
    """Listening scaler on active days only: p99.9 clip then mean/std, fit rows only."""
    # Pass 1: bounded quantile sample of active-day values.
    sample, rng = [], np.random.default_rng(SEED)
    for block in _listening_block_iter(rows):
        active = block[:, :, -1] > 0
        vals = block[:, :, :-1][active]
        if len(vals):
            keep = min(len(vals), max(1, QUANTILE_SAMPLE // 50))
            sample.append(vals[rng.choice(len(vals), keep, replace=False)]
                          if len(vals) > keep else vals)
    values = np.concatenate(sample, axis=0).astype(np.float64) if sample else np.empty((0, 7))
    if len(values) == 0:
        raise ValueError("fit sample has no listening records")
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError("expected finite non-negative log1p values")
    upper = np.quantile(values, 0.999, axis=0)
    # Pass 2: mean/std over all active-day values, clipped, in blocks.
    n = np.zeros(7); s = np.zeros(7); ss = np.zeros(7)
    for block in _listening_block_iter(rows):
        active = block[:, :, -1] > 0
        vals = np.minimum(block[:, :, :-1][active], upper).astype(np.float64)
        if len(vals):
            n += len(vals); s += vals.sum(0); ss += (vals ** 2).sum(0)
    mean = s / np.maximum(n, 1)
    std = np.sqrt(np.maximum(ss / np.maximum(n, 1) - mean ** 2, 0))
    std[std < 1e-6] = 1.0
    return {"encoding": "log1p", "upper": upper.tolist(),
            "mean": mean.tolist(), "std": std.tolist()}


def transform_sequence(sequence, state):
    """Apply the listening scaler; missing (inactive) days stay zero, flag preserved."""
    x = np.array(sequence, dtype=np.float32, copy=True)
    active = x[:, :, -1] > 0
    values = x[:, :, :-1]
    values[active] = ((np.minimum(values[active], state["upper"]) - np.asarray(state["mean"]))
                      / np.asarray(state["std"]))
    values[~active] = 0
    assert np.isfinite(x).all()
    return x

In [ ]:
def _combined_block_iter(rows, block=20000):
    arr = SEQ[("lstm", "train")]
    for start in range(0, len(rows), block):
        idx = rows[start:start + block]
        yield np.asarray(arr[idx, -N_DAYS:, :], dtype=np.float32)


def fit_transaction_scaler(rows, specs):
    """Event/state scaler per channel using its own mask; flags are not scaled.

    signed_log1p channels keep their sign and use a bilateral p0.1/p99.9 clip; count-like
    channels clip at [0, p99.9]. Statistics come from fit rows only.
    """
    channels = [item["name"] for item in specs]
    # Pass 1: bounded quantile sample per scaled channel.
    samples = {item["name"]: [] for item in specs if not (specs.index(item) < 8 or item["encoding"] == "flag")}
    rng = np.random.default_rng(SEED)
    for seqblock in _combined_block_iter(rows):
        for j, item in enumerate(specs):
            if j < 8 or item["encoding"] == "flag":
                continue
            mask = seqblock[:, :, channels.index(item["mask"])] > 0
            vals = seqblock[:, :, j][mask]
            if len(vals):
                keep = min(len(vals), max(1, QUANTILE_SAMPLE // 50))
                samples[item["name"]].append(
                    vals[rng.choice(len(vals), keep, replace=False)] if len(vals) > keep else vals)
    limits = {}
    for item in specs:
        name = item["name"]
        if name not in samples:
            continue
        vals = np.concatenate(samples[name]).astype(np.float64) if samples[name] else np.empty(0)
        if not len(vals):
            limits[name] = None
        else:
            lower = float(np.quantile(vals, 0.001)) if item["encoding"] == "signed_log1p" else 0.0
            limits[name] = (lower, float(np.quantile(vals, 0.999)))
    # Pass 2: mean/std of clipped values in blocks.
    acc = {name: [0.0, 0.0, 0.0] for name in samples}  # n, sum, sumsq
    for seqblock in _combined_block_iter(rows):
        for j, item in enumerate(specs):
            name = item["name"]
            if name not in samples or limits[name] is None:
                continue
            mask = seqblock[:, :, channels.index(item["mask"])] > 0
            vals = np.clip(seqblock[:, :, j][mask], limits[name][0], limits[name][1]).astype(np.float64)
            if len(vals):
                acc[name][0] += len(vals); acc[name][1] += vals.sum(); acc[name][2] += (vals ** 2).sum()
    state = {"channels": channels, "parameters": {}}
    for name in samples:
        if limits[name] is None:
            state["parameters"][name] = {"lower": None, "upper": None,
                                         "mean": 0.0, "std": 1.0, "empty_fit": True}
        else:
            n, s, ss = acc[name]
            mean = s / max(n, 1); std = np.sqrt(max(ss / max(n, 1) - mean ** 2, 0))
            state["parameters"][name] = {"lower": limits[name][0], "upper": limits[name][1],
                                         "mean": float(mean), "std": float(max(std, 1e-6)),
                                         "empty_fit": False}
    return state


def transform_combined(sequence, listening_state, transaction_state, specs):
    """Apply listening scaler to the first 8 channels and the transaction scaler to the rest."""
    x = np.array(sequence, dtype=np.float32, copy=True)
    x[:, :, :8] = transform_sequence(sequence[:, :, :8], listening_state)
    channels = transaction_state["channels"]
    for j, item in enumerate(specs):
        if j < 8 or item["encoding"] == "flag":
            continue
        mask = sequence[:, :, channels.index(item["mask"])] > 0
        vals = sequence[:, :, j][mask]
        p = transaction_state["parameters"][item["name"]]
        if p["lower"] is not None:
            vals = np.clip(vals, p["lower"], p["upper"])
        x[:, :, j] = 0
        x[:, :, j][mask] = (vals - p["mean"]) / p["std"]
    assert np.isfinite(x).all()
    return x

In [ ]:
def numeric_values(frame, columns):
    values = frame[columns].to_numpy(dtype=np.float64, copy=True)
    values[~np.isfinite(values)] = np.nan
    return values


def category_values(series):
    return series.astype("string").fillna("__MISSING__").to_numpy(dtype=str)


def fit_tabular_preprocessor(frame, features, categorical):
    """Median impute, signed log1p, p0.5/p99.5 clip, standardize; one-hot with fit levels."""
    numeric = [c for c in features if c not in categorical]
    raw = numeric_values(frame, numeric)
    median = np.array([np.nanmedian(col) if np.isfinite(col).any() else 0.0 for col in raw.T])
    x = np.where(np.isnan(raw), median, raw)
    x = np.sign(x) * np.log1p(np.abs(x))
    lower, upper = np.quantile(x, [0.005, 0.995], axis=0)
    x = np.clip(x, lower, upper)
    mean, std = x.mean(axis=0), x.std(axis=0)
    std[std < 1e-6] = 1.0
    levels = {c: sorted(set(category_values(frame[c]))) for c in categorical}
    return {"numeric": numeric, "categorical": list(categorical),
            "median": median.tolist(), "lower": lower.tolist(), "upper": upper.tolist(),
            "mean": mean.tolist(), "std": std.tolist(), "levels": levels}


def transform_tabular(frame, state):
    raw = numeric_values(frame, state["numeric"])
    x = np.where(np.isnan(raw), np.asarray(state["median"]), raw)
    x = np.sign(x) * np.log1p(np.abs(x))
    x = np.clip(x, state["lower"], state["upper"])
    pieces = [(x - np.asarray(state["mean"])) / np.asarray(state["std"])]
    for col in state["categorical"]:
        values = category_values(frame[col])
        pieces.append(np.column_stack([values == level for level in state["levels"][col]]))
    out = np.concatenate(pieces, axis=1).astype(np.float32)
    assert np.isfinite(out).all()
    return out

In [ ]:
# Daily context channels (causal), used only by sequence configs that request it.
EXTRA_CHANNELS = ["days_since_active_30", "active_days_7", "active_days_30", "day_position"]

def add_daily_context(sequence, reference_days=N_DAYS, channels=None):
    x = np.asarray(sequence, dtype=np.float32)
    if x.ndim != 3 or not (1 <= x.shape[1] <= reference_days):
        raise ValueError("expected (users, days, channels) within the window")
    channels = channels or LISTEN_CHANNELS
    active = x[:, :, channels.index("active")]
    if not np.isfinite(x).all() or not np.isin(active, [0, 1]).all():
        raise ValueError("non-finite channels or invalid active flag")
    n, days = active.shape
    position = np.arange(days, dtype=np.int32)[None, :]
    last_active = np.maximum.accumulate(np.where(active > 0, position, -reference_days), axis=1)
    gap = np.minimum(position - last_active, 30) / 30.0
    cumulative = np.concatenate([np.zeros((n, 1), np.float32), np.cumsum(active, axis=1)], axis=1)
    end = np.arange(1, days + 1)
    recent_7 = (cumulative[:, end] - cumulative[:, np.maximum(end - 7, 0)]) / 7.0
    recent_30 = (cumulative[:, end] - cumulative[:, np.maximum(end - 30, 0)]) / 30.0
    day_position = np.broadcast_to(position / max(reference_days - 1, 1), (n, days))
    extra = np.stack([gap, recent_7, recent_30, day_position], axis=-1)
    return np.concatenate([x, extra], axis=-1).astype(np.float32)

In [ ]:
# ---- Phase scalers: fit once per training-row set, reuse across trials ----
# A "phase" names the rows a scaler is fit on. We fit on fit (for Optuna / development
# search), on development = February (development refit) and on train = February + March
# (final refit used for the April test).
PHASE_ROWS = {"fit": SPLIT["fit"], "development": DEV_ROWS,
              "train": np.arange(len(train_df))}
if SMOKE_USERS is not None:
    PHASE_ROWS["train"] = np.sort(np.concatenate(list(SPLIT.values())))

_scaler_cache = {}

def phase_scalers(phase):
    """Return (listening_scaler, transaction_scaler, tabular_preprocessor) for a phase."""
    if phase not in _scaler_cache:
        rows = PHASE_ROWS[phase]
        listen = fit_sequence_scaler(rows)
        tx = fit_transaction_scaler(rows, TX_SPECS)
        tab = fit_tabular_preprocessor(train_df.iloc[rows], FEATURES, CATEGORICAL)
        _scaler_cache[phase] = (listen, tx, tab)
    return _scaler_cache[phase]

In [ ]:
# ---- Transformed batch access by original row index -----------------------
def sequence_batch(rows, source, phase, daily_context=False, dataset="train"):
    """Transformed sequence block for the given rows and model source.

    source: 'listening' (8ch), 'transaction' (12ch), 'combined' (20ch).
    Scaler parameters come from the named phase; rows index the original tensor.
    """
    listen, tx, _ = phase_scalers(phase)
    rows = np.asarray(rows)
    if source == "listening":
        raw = np.asarray(SEQ[("hybrid", dataset)][rows, -N_DAYS:, :], dtype=np.float32)
        x = transform_sequence(raw, listen)
        channels = list(LISTEN_CHANNELS)
    else:
        raw = np.asarray(SEQ[("lstm", dataset)][rows, -N_DAYS:, :], dtype=np.float32)
        combined = transform_combined(raw, listen, tx, TX_SPECS)
        if source == "transaction":
            return combined[:, :, 8:], list(TX_CHANNELS)
        x = combined
        channels = list(LSTM_CHANNELS)
    if daily_context:
        x = add_daily_context(x, channels=channels)
        channels = channels + EXTRA_CHANNELS
    return x, channels


def tabular_batch(rows, phase, dataset="train"):
    _, _, tab = phase_scalers(phase)
    frame = (train_df if dataset == "train" else test_df).iloc[np.asarray(rows)]
    return transform_tabular(frame, tab)


class SequenceLoader(tf.keras.utils.Sequence):
    """Keras loader that transforms sequence (and optional tabular) blocks on demand.

    Order is fixed (no shuffling across epochs) so predictions keep row order; training
    shuffling is handled by Keras via the index permutation we build per epoch.
    """
    def __init__(self, rows, source, phase, batch_size, labels=None,
                 daily_context=False, hybrid=False, dataset="train", shuffle=False):
        self.rows = np.asarray(rows)
        self.source, self.phase = source, phase
        self.batch_size = int(batch_size)
        self.labels = None if labels is None else np.asarray(labels)
        self.daily_context, self.hybrid = daily_context, hybrid
        self.dataset, self.shuffle = dataset, shuffle
        self._order = np.arange(len(self.rows))
        self._rng = np.random.default_rng(SEED)

    def __len__(self):
        return int(np.ceil(len(self.rows) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            self._rng.shuffle(self._order)

    def __getitem__(self, i):
        sel = self._order[i * self.batch_size:(i + 1) * self.batch_size]
        rows = self.rows[sel]
        seq, _ = sequence_batch(rows, self.source, self.phase,
                                daily_context=self.daily_context, dataset=self.dataset)
        if self.hybrid:
            tab = tabular_batch(rows, self.phase, dataset=self.dataset)
            x = {"tabular": tab, "sequence": seq}
        else:
            x = seq
        if self.labels is None:
            return x
        return x, self.labels[sel]

## 5. Metrics and confusion-matrix helpers

One evaluation function takes labels, probabilities, a model id and an **already-fixed**
threshold; it never trains or optimises anything. The threshold is chosen by maximum F1
of churn on **tune** with a deterministic tie rule (prefer the higher threshold). Both
classes are required for AUC; `zero_division=0` and empty denominators are handled.

In [ ]:
EPS = 1e-7

def select_threshold(y_true, proba):
    """Threshold maximising churn F1; ties prefer the higher threshold (deterministic)."""
    y_true = np.asarray(y_true); proba = np.asarray(proba)
    grid = np.linspace(0.01, 0.99, 99)
    best_t, best_f1 = 0.5, -1.0
    for t in grid:
        f1 = f1_score(y_true, (proba >= t).astype(int), zero_division=0)
        if f1 >= best_f1:            # >= so ties keep the higher threshold
            best_f1, best_t = f1, float(t)
    return best_t


def evaluate(y_true, proba, model_id, threshold):
    """Full metric row at a fixed threshold. No training, no threshold search here."""
    y_true = np.asarray(y_true).astype(int)
    proba = np.clip(np.asarray(proba, dtype=np.float64), EPS, 1 - EPS)
    pred = (proba >= threshold).astype(int)
    two_classes = len(np.unique(y_true)) == 2
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    return {
        "model_id": model_id,
        "log_loss": log_loss(y_true, proba, labels=[0, 1]),
        "roc_auc": roc_auc_score(y_true, proba) if two_classes else np.nan,
        "average_precision": average_precision_score(y_true, proba) if two_classes else np.nan,
        "brier": brier_score_loss(y_true, proba),
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision_churn": precision_score(y_true, pred, zero_division=0),
        "recall_churn": recall_score(y_true, pred, zero_division=0),
        "specificity": specificity,
        "f1_churn": f1_score(y_true, pred, zero_division=0),
        "f1_macro": f1_score(y_true, pred, average="macro", zero_division=0),
        "mcc": matthews_corrcoef(y_true, pred) if two_classes else 0.0,
        "threshold": float(threshold),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "support_0": int((y_true == 0).sum()), "support_1": int((y_true == 1).sum()),
        "actual_churn_rate": float(y_true.mean()),
        "predicted_churn_rate": float(pred.mean()),
    }


def metrics_at_half(y_true, proba, model_id):
    return evaluate(y_true, proba, model_id, 0.5)

In [ ]:
# ---- Confusion-matrix figures (counts + row-normalised) -------------------
CLASS_LABELS = ["Permanece (0)", "Churn (1)"]

def _draw_confusion(ax_counts, ax_norm, y_true, proba, threshold, title):
    y_true = np.asarray(y_true).astype(int)
    pred = (np.asarray(proba) >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    row_sums = cm.sum(axis=1, keepdims=True)
    norm = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float),
                     where=row_sums != 0) * 100.0
    # Counts panel (own colour scale).
    im0 = ax_counts.imshow(cm, cmap="Blues")
    ax_counts.figure.colorbar(im0, ax=ax_counts, fraction=0.046, pad=0.04)
    names = ["TN", "FP", "FN", "TP"]
    for (r, c), v in np.ndenumerate(cm):
        ax_counts.text(c, r, f"{names[r*2+c]}\n{v:d}", ha="center", va="center",
                       color="white" if v > cm.max() / 2 else "black", fontsize=10)
    ax_counts.set_title(f"{title}\ncounts @ thr={threshold:.2f}", fontsize=10)
    # Row-normalised panel (shared 0-100 scale).
    im1 = ax_norm.imshow(norm, cmap="Blues", vmin=0, vmax=100)
    ax_norm.figure.colorbar(im1, ax=ax_norm, fraction=0.046, pad=0.04)
    for (r, c), v in np.ndenumerate(norm):
        ax_norm.text(c, r, f"{v:.1f}%", ha="center", va="center",
                     color="white" if v > 50 else "black", fontsize=10)
    ax_norm.set_title(f"{title}\nrow-normalised %", fontsize=10)
    for ax in (ax_counts, ax_norm):
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(CLASS_LABELS, rotation=20, ha="right", fontsize=8)
        ax.set_yticklabels(CLASS_LABELS, fontsize=8)
        ax.set_xlabel("Predicción"); ax.set_ylabel("Clase real")


def save_confusion(y_true, proba, threshold, model_id, out_dir, tag="holdout"):
    """Save a single high-resolution confusion figure (counts + normalised) as PNG+SVG."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    _draw_confusion(axes[0], axes[1], y_true, proba, threshold, f"{model_id} ({tag})")
    fig.tight_layout()
    fig.savefig(out_dir / f"confusion_{model_id}_{tag}.png", dpi=200, bbox_inches="tight")
    fig.savefig(out_dir / f"confusion_{model_id}_{tag}.svg", bbox_inches="tight")
    return fig


def gallery_confusion(entries, thresholds, out_dir, tag="holdout", per_page=3):
    """Paginated gallery, up to `per_page` models per figure (counts + normalised cols)."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    items = list(entries.items())
    figs = []
    for page in range(0, len(items), per_page):
        chunk = items[page:page + per_page]
        fig, axes = plt.subplots(len(chunk), 2, figsize=(9, 4 * len(chunk)), squeeze=False)
        for r, (model_id, (y_true, proba)) in enumerate(chunk):
            _draw_confusion(axes[r, 0], axes[r, 1], y_true, proba,
                            thresholds[model_id], f"{model_id} ({tag})")
        fig.tight_layout()
        fig.savefig(out_dir / f"gallery_{tag}_{page // per_page}.png", dpi=160, bbox_inches="tight")
        figs.append(fig)
    return figs

In [ ]:
def save_classification_report(y_true, proba, threshold, model_id, out_dir):
    """Full classification_report (per class + macro/weighted) as JSON and text."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    pred = (np.asarray(proba) >= threshold).astype(int)
    rep_dict = classification_report(y_true, pred, labels=[0, 1],
                                     target_names=CLASS_LABELS, output_dict=True,
                                     zero_division=0)
    rep_text = classification_report(y_true, pred, labels=[0, 1],
                                     target_names=CLASS_LABELS, zero_division=0)
    with (out_dir / f"report_{model_id}.json").open("w") as f:
        json.dump(rep_dict, f, indent=2)
    (out_dir / f"report_{model_id}.txt").write_text(rep_text)
    return rep_dict

## 6. Model registry

A registry stores, per model id, how to fit, predict, save and load a predictor, plus its
inputs, best hyper-parameters, trial accounting, timings and the chosen threshold. A load
function reconstructs each predictor without relying on notebook global state. Blends
reference their frozen components and weights.

In [ ]:
# Everything a model produced lives here; nothing downstream reads training globals.
REGISTRY = {}   # model_id -> dict(record)
PROBA = {}      # model_id -> {"tune": .., "holdout": .., "test": ..} probability arrays

def register(model_id, **record):
    REGISTRY[model_id] = {"model_id": model_id, **record}
    return REGISTRY[model_id]


def set_proba(model_id, part, proba):
    PROBA.setdefault(model_id, {})[part] = np.clip(np.asarray(proba, dtype=np.float64),
                                                   EPS, 1 - EPS)


def rows_of(part):
    return SPLIT[part]


def y_of(part):
    return y_all[SPLIT[part]]

In [ ]:
# ---- Optuna study helpers (SQLite persistence, resumable budgets) ---------
OPTUNA_DIR = OUT_DIR / "optuna"
OPTUNA_DIR.mkdir(parents=True, exist_ok=True)
STORAGE = f"sqlite:///{(OPTUNA_DIR / 'studies.sqlite3').as_posix()}"

# A fingerprint of the data contract + split; a study is not resumed if it changes.
CONTRACT = {
    "dataset_version": SEQ_META["dataset_version"],
    "hash_train_feb": SEQ_META["hash_train_feb"], "hash_train": SEQ_META["hash_train"],
    "hash_test": SEQ_META["hash_test"], "cutoffs": SEQ_META["cutoffs"],
    "split_protocol": "cross_month_feb_to_mar",
    "split_sizes": {k: int(len(v)) for k, v in SPLIT.items()},
    "n_days": N_DAYS, "smoke": SMOKE_TEST,
    "libs": {k: ENV[k] for k in ("optuna", "tensorflow", "sklearn")},
}
CONTRACT_HASH = hashlib.sha1(json.dumps(CONTRACT, sort_keys=True).encode()).hexdigest()[:12]


def get_study(model_id):
    """Create or resume a persistent study; reject a different data/split contract."""
    name = f"{model_id}_{CONTRACT_HASH}"
    study = optuna.create_study(
        study_name=name, storage=STORAGE, load_if_exists=True,
        direction="minimize", sampler=optuna.samplers.TPESampler(
            seed=SEED, n_startup_trials=max(1, min(3, TRIALS[model_id] // 2))),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=1),
    )
    study.set_user_attr("contract", CONTRACT)
    return study


def remaining_trials(study, total):
    """Count all attempted trials, including failed/interrupted ones, against budget."""
    done = sum(t.state != optuna.trial.TrialState.WAITING for t in study.trials)
    return max(0, total - done)


def run_study(model_id, objective, total_trials, enqueue=None):
    study = get_study(model_id)
    if enqueue:
        existing = {json.dumps(t.params, sort_keys=True) for t in study.trials}
        for params in enqueue:
            if json.dumps(params, sort_keys=True) not in existing:
                study.enqueue_trial(params, skip_if_exists=True)
    n = remaining_trials(study, total_trials)
    print(f"{model_id}: budget={total_trials}, remaining={n}", flush=True)

    def objective_with_progress(trial):
        print(f"{model_id}: starting trial #{trial.number}", flush=True)
        return objective(trial)

    def report_trial(study, trial):
        seconds = trial.duration.total_seconds() if trial.duration else 0.0
        print(f"{model_id}: trial #{trial.number} {trial.state.name} "
              f"| log-loss={trial.value} | {seconds:.1f}s", flush=True)

    if n > 0:
        study.optimize(objective_with_progress, n_trials=n, gc_after_trial=True,
                       show_progress_bar=True, callbacks=[report_trial])
    if not any(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials):
        raise RuntimeError(f"{model_id}: study has no completed trial; cannot select params.")
    df = study.trials_dataframe()
    df.to_csv(OPTUNA_DIR / f"{model_id}_trials.csv", index=False)
    with (OPTUNA_DIR / f"{model_id}_best_params.json").open("w") as f:
        json.dump(study.best_params, f, indent=2)
    counts = {
        "complete": sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials),
        "pruned": sum(t.state == optuna.trial.TrialState.PRUNED for t in study.trials),
        "failed": sum(t.state == optuna.trial.TrialState.FAIL for t in study.trials),
    }
    if counts["complete"] == 0:
        raise RuntimeError(f"{model_id}: study has no completed trial; cannot select params.")
    print(f"{model_id}: best tune log-loss={study.best_value:.5f} | trials {counts}")
    return study, counts

## 8. Sequence studies (M5, M7, M8), MLP backbone (M9) and hybrid (M10)

M5 reads listening (8 channels), M7 the 12 transaction channels, M8 the 20 combined.
All are real recurrent models with a classifier head; `pool` means concatenate the final
state, the mean and the max of the last recurrent layer's outputs. Sigmoid output in
float32, gradient clipping, logits initialised with the fit prevalence, cuDNN-compatible
(no recurrent dropout).

M9 is the tabular MLP, trained here **only as M10's backbone**; it is not compared. M10
freezes the winning M9 and trains a new recurrent correction starting at zero; its first
prediction matches M9 within tolerance. If training does not beat `stop`, the zero
correction is restored and the row is labelled "equivalent to MLP" (not counted as
recurrent contribution). The threshold selection and value of every trial is tune log-loss
with restored best weights.


In [ ]:
# Development-phase labels, fit only on `fit`.
y_fit, y_stop, y_tune = y_of("fit"), y_of("stop"), y_of("tune")


In [ ]:
def prior_bias(rate):
    rate = float(np.clip(rate, 1e-6, 1 - 1e-6))
    return tf.keras.initializers.Constant(np.log(rate / (1 - rate)))

def _compile(model, lr):
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0),
                  loss="binary_crossentropy")
    return model

def sequence_representation(inputs, units, second_units, dropout, pool):
    """One or two recurrent layers; optional final+mean+max pooling of last layer."""
    x = inputs
    if second_units:
        x = tf.keras.layers.LSTM(units, return_sequences=True, name="lstm_1")(x)
        if pool:
            states, last, _ = tf.keras.layers.LSTM(
                second_units, return_sequences=True, return_state=True, name="lstm_2")(x)
            x = tf.keras.layers.Concatenate()([
                last, tf.keras.layers.GlobalAveragePooling1D()(states),
                tf.keras.layers.GlobalMaxPooling1D()(states)])
        else:
            x = tf.keras.layers.LSTM(second_units, name="lstm_2")(x)
    else:
        if pool:
            states, last, _ = tf.keras.layers.LSTM(
                units, return_sequences=True, return_state=True, name="lstm_1")(x)
            x = tf.keras.layers.Concatenate()([
                last, tf.keras.layers.GlobalAveragePooling1D()(states),
                tf.keras.layers.GlobalMaxPooling1D()(states)])
        else:
            x = tf.keras.layers.LSTM(units, name="lstm_1")(x)
    return tf.keras.layers.Dropout(dropout)(x)

def build_sequence_model(input_shape, rate, units, second_units, dropout, lr, pool):
    inputs = tf.keras.Input(shape=input_shape, name="sequence")
    x = sequence_representation(inputs, units, second_units, dropout, pool)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid", dtype="float32",
                                    bias_initializer=prior_bias(rate))(x)
    return _compile(tf.keras.Model(inputs, outputs), lr)

def build_tabular_model(n_features, rate, units_1, units_2, dropout, lr):
    inputs = tf.keras.Input(shape=(n_features,), name="tabular")
    x = tf.keras.layers.Dense(units_1, activation="relu")(inputs)
    x = tf.keras.layers.Dropout(dropout)(x)
    if units_2:
        x = tf.keras.layers.Dense(units_2, activation="relu")(x)
    logits = tf.keras.layers.Dense(1, bias_initializer=prior_bias(rate),
                                   name="tabular_logits")(x)
    outputs = tf.keras.layers.Activation("sigmoid", dtype="float32")(logits)
    return _compile(tf.keras.Model(inputs, outputs, name="mlp_tabular"), lr)

def build_hybrid_model(backbone, n_features, seq_shape, units, second_units, dropout, lr, pool):
    tab = tf.keras.Input(shape=(n_features,), name="tabular")
    seq = tf.keras.Input(shape=seq_shape, name="sequence")
    x = sequence_representation(seq, units, second_units, dropout, pool)
    correction = tf.keras.layers.Dense(1, kernel_initializer="zeros",
                                       bias_initializer="zeros", name="sequence_correction")(x)
    logits = tf.keras.layers.Add()([backbone(tab, training=False), correction])
    outputs = tf.keras.layers.Activation("sigmoid", dtype="float32")(logits)
    return _compile(tf.keras.Model({"tabular": tab, "sequence": seq}, outputs), lr)

def dl_callbacks():
    return [tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                                 patience=2, min_lr=1e-5),
            tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=DL_PATIENCE,
                                             restore_best_weights=True)]

class OptunaPruneCB(tf.keras.callbacks.Callback):
    def __init__(self, trial):
        super().__init__(); self.trial = trial
    def on_epoch_end(self, epoch, logs=None):
        self.trial.report(logs["val_loss"], step=epoch)
        if self.trial.should_prune():
            raise optuna.TrialPruned()

def predict_loader(model, loader):
    return np.clip(model.predict(loader, verbose=0).ravel(), EPS, 1 - EPS)

In [ ]:
# Shared sequence hyper-parameter space (M5/M7/M8).
def suggest_seq_params(trial):
    n_layers = trial.suggest_int("n_layers", 1, 2)
    params = {
        "units": trial.suggest_categorical("units", [32, 64, 128]),
        "second_units": (trial.suggest_categorical("second_units", [16, 32, 64])
                         if n_layers == 2 else None),
        "dropout": trial.suggest_float("dropout", 0.0, 0.4),
        "lr": trial.suggest_float("lr", 1e-4, 3e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [128, 256, 512, 1024]),
        "pool": trial.suggest_categorical("pool", [True, False]),
    }
    return n_layers, params

SEQ_ENQUEUE = [{"n_layers": 1, "units": 32, "dropout": 0.1, "lr": 1e-3,
                "batch_size": 128, "pool": True}]

SOURCE_OF = {"M5": "listening", "M7": "transaction", "M8": "combined"}

def _seq_loaders(model_id, phase, batch_size, parts=("fit", "stop", "tune")):
    src = SOURCE_OF[model_id]
    out = {}
    for part in parts:
        shuffle = part == "fit"
        out[part] = SequenceLoader(rows_of(part), src, phase, batch_size,
                                   labels=y_of(part), shuffle=shuffle)
    return out

def _seq_input_shape(model_id):
    probe, channels = sequence_batch(rows_of("fit")[:2], SOURCE_OF[model_id], "fit")
    return probe.shape[1:], channels

def make_seq_objective(model_id):
    input_shape, channels = _seq_input_shape(model_id)
    def objective(trial):
        n_layers, params = suggest_seq_params(trial)
        tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)
        model = build_sequence_model(input_shape, FIT_RATE, params["units"],
                                     params["second_units"], params["dropout"],
                                     params["lr"], params["pool"])
        loaders = _seq_loaders(model_id, "fit", params["batch_size"])
        try:
            model.fit(loaders["fit"], validation_data=loaders["stop"],
                      epochs=DL_MAX_EPOCHS, verbose=TRAIN_VERBOSE,
                      callbacks=dl_callbacks() + [OptunaPruneCB(trial)])
        except tf.errors.ResourceExhaustedError:
            raise optuna.TrialPruned()   # OOM: explicit prune, do not fake a bad score
        p_tune = predict_loader(model, loaders["tune"])
        del model
        return log_loss(y_tune, p_tune, labels=[0, 1])
    return objective, channels

def run_sequence(model_id):
    t0 = time.perf_counter()
    objective, channels = make_seq_objective(model_id)
    study, counts = run_study(model_id, objective, TRIALS[model_id], enqueue=SEQ_ENQUEUE)
    bp = study.best_params
    n_layers = bp["n_layers"]
    second = bp.get("second_units") if n_layers == 2 else None
    input_shape, _ = _seq_input_shape(model_id)
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)
    model = build_sequence_model(input_shape, FIT_RATE, bp["units"], second,
                                 bp["dropout"], bp["lr"], bp["pool"])
    loaders = _seq_loaders(model_id, "fit", bp["batch_size"])
    hist = model.fit(loaders["fit"], validation_data=loaders["stop"],
                     epochs=DL_MAX_EPOCHS, verbose=TRAIN_VERBOSE, callbacks=dl_callbacks())
    best_epoch = int(np.argmin(hist.history["val_loss"])) + 1
    set_proba(model_id, "tune", predict_loader(model, loaders["tune"]))
    hold_loader = SequenceLoader(rows_of("holdout"), SOURCE_OF[model_id], "fit",
                                 INFER_BATCH, labels=None)
    set_proba(model_id, "holdout", predict_loader(model, hold_loader))
    register(model_id, family="sequence", input=SOURCE_OF[model_id],
             n_channels=len(channels), channels=channels,
             n_parameters=int(model.count_params()), best_params=bp, trials=counts,
             fit_seconds=time.perf_counter() - t0, best_epoch=best_epoch,
             history=hist.history, estimator=model, batch_size=bp["batch_size"],
             pool=bp["pool"])
    print(f"{model_id}: params={model.count_params()} best_epoch={best_epoch}")

In [ ]:
# Build the three pure LSTMs.
for _mid in ("M5", "M7", "M8"):
    run_sequence(_mid)

In [ ]:
# ---- M9: tabular MLP (own Optuna; backbone for M10) ----------------------
T_fit = tabular_batch(rows_of("fit"), "fit")
T_stop = tabular_batch(rows_of("stop"), "fit")
T_tune = tabular_batch(rows_of("tune"), "fit")
N_TAB = T_fit.shape[1]

def suggest_mlp_params(trial):
    n_layers = trial.suggest_int("n_layers", 1, 2)
    return {
        "units_1": trial.suggest_categorical("units_1", [32, 64, 128]),
        "units_2": (trial.suggest_categorical("units_2", [32, 64, 128])
                    if n_layers == 2 else None),
        "dropout": trial.suggest_float("dropout", 0.0, 0.4),
        "lr": trial.suggest_float("lr", 1e-4, 3e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [256, 512, 1024]),
    }

def _fit_mlp(params):
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)
    model = build_tabular_model(N_TAB, FIT_RATE, params["units_1"], params["units_2"],
                                params["dropout"], params["lr"])
    hist = model.fit(T_fit, y_fit, validation_data=(T_stop, y_stop),
                     epochs=DL_MAX_EPOCHS, batch_size=params["batch_size"], verbose=TRAIN_VERBOSE,
                     callbacks=dl_callbacks())
    return model, hist

def M9_objective(trial):
    params = suggest_mlp_params(trial)
    model, hist = _fit_mlp(params)
    p = np.clip(model.predict(T_tune, batch_size=INFER_BATCH, verbose=0).ravel(), EPS, 1 - EPS)
    trial.set_user_attr("best_epoch", int(np.argmin(hist.history["val_loss"])) + 1)
    del model
    return log_loss(y_tune, p, labels=[0, 1])

def run_M9():
    t0 = time.perf_counter()
    study, counts = run_study("M9", M9_objective, TRIALS["M9"], enqueue=[{
        "n_layers": 2, "units_1": 64, "units_2": 32, "dropout": 0.15,
        "lr": 1e-3, "batch_size": 512}])
    bp = study.best_params
    params = {"units_1": bp["units_1"],
              "units_2": bp.get("units_2") if bp["n_layers"] == 2 else None,
              "dropout": bp["dropout"], "lr": bp["lr"], "batch_size": bp["batch_size"]}
    model, hist = _fit_mlp(params)
    best_epoch = int(np.argmin(hist.history["val_loss"])) + 1
    set_proba("M9", "tune", np.clip(model.predict(T_tune, batch_size=INFER_BATCH,
                                                   verbose=0).ravel(), EPS, 1 - EPS))
    T_hold = tabular_batch(rows_of("holdout"), "fit")
    set_proba("M9", "holdout", np.clip(model.predict(T_hold, batch_size=INFER_BATCH,
                                                      verbose=0).ravel(), EPS, 1 - EPS))
    register("M9", family="mlp", input="table", n_channels=N_TAB,
             n_parameters=int(model.count_params()), best_params=bp, trials=counts,
             fit_seconds=time.perf_counter() - t0, best_epoch=best_epoch,
             history=hist.history, estimator=model, mlp_params=params)
    print("M9 params:", model.count_params(), "best_epoch:", best_epoch)

run_M9()

In [ ]:
# ---- M10: hybrid MLP backbone (frozen) + listening LSTM correction -------
def _frozen_backbone(mlp):
    backbone = tf.keras.Model(mlp.input, mlp.get_layer("tabular_logits").output,
                              name="frozen_tabular_backbone")
    backbone.trainable = False
    return backbone

def _hybrid_inputs(part, batch_size, shuffle=False):
    return SequenceLoader(rows_of(part), "listening", "fit", batch_size,
                          labels=y_of(part), hybrid=True, shuffle=shuffle)

def M10_objective(trial):
    # Backbone is the trained M9, shared and frozen across trials.
    mlp = REGISTRY["M9"]["estimator"]
    backbone = _frozen_backbone(mlp)
    frozen_weights = backbone.get_weights()
    n_layers, params = suggest_seq_params(trial)
    seq_shape, _ = _seq_input_shape("M5")   # listening shape
    tf.keras.utils.set_random_seed(SEED)
    model = build_hybrid_model(backbone, N_TAB, seq_shape, params["units"],
                               params["second_units"], params["dropout"],
                               params["lr"], params["pool"])
    loaders = {p: _hybrid_inputs(p, params["batch_size"], shuffle=(p == "fit"))
               for p in ("fit", "stop", "tune")}
    try:
        model.fit(loaders["fit"], validation_data=loaders["stop"],
                  epochs=DL_MAX_EPOCHS, verbose=TRAIN_VERBOSE,
                  callbacks=dl_callbacks() + [OptunaPruneCB(trial)])
    except tf.errors.ResourceExhaustedError:
        raise optuna.TrialPruned()
    # Backbone must stay frozen.
    for before, after in zip(frozen_weights, backbone.get_weights()):
        np.testing.assert_array_equal(before, after)
    p_tune = predict_loader(model, loaders["tune"])
    del model
    return log_loss(y_tune, p_tune, labels=[0, 1])

def run_M10():
    t0 = time.perf_counter()
    study, counts = run_study("M10", M10_objective, TRIALS["M10"], enqueue=SEQ_ENQUEUE)
    bp = study.best_params
    n_layers = bp["n_layers"]; second = bp.get("second_units") if n_layers == 2 else None
    mlp = REGISTRY["M9"]["estimator"]
    backbone = _frozen_backbone(mlp)
    frozen_weights = backbone.get_weights()
    seq_shape, _ = _seq_input_shape("M5")
    tf.keras.utils.set_random_seed(SEED)
    model = build_hybrid_model(backbone, N_TAB, seq_shape, bp["units"], second,
                               bp["dropout"], bp["lr"], bp["pool"])
    initial_weights = model.get_weights()
    # The zero correction leaves the frozen MLP as the only signal. The two
    # TensorFlow prediction graphs can differ slightly in float32 rounding.
    stop_hybrid = _hybrid_inputs("stop", INFER_BATCH)
    np.testing.assert_array_equal(stop_hybrid.rows, rows_of("stop"))
    for weight in model.get_layer("sequence_correction").get_weights():
        assert np.count_nonzero(weight) == 0
    mlp_stop = np.clip(mlp.predict(T_stop, batch_size=INFER_BATCH, verbose=0).ravel(), EPS, 1 - EPS)
    initial_hybrid_stop = predict_loader(model, stop_hybrid)
    np.testing.assert_allclose(initial_hybrid_stop, mlp_stop, rtol=0, atol=2e-4)
    initial_loss = log_loss(y_stop, initial_hybrid_stop, labels=[0, 1])
    loaders = {p: _hybrid_inputs(p, bp["batch_size"], shuffle=(p == "fit"))
               for p in ("fit", "stop", "tune")}
    hist = model.fit(loaders["fit"], validation_data=loaders["stop"],
                     epochs=DL_MAX_EPOCHS, verbose=TRAIN_VERBOSE, callbacks=dl_callbacks())
    trained_loss = log_loss(y_stop, predict_loader(model, loaders["stop"]), labels=[0, 1])
    kept = bool(trained_loss < initial_loss - 1e-5)
    best_epoch = int(np.argmin(hist.history["val_loss"])) + 1
    if not kept:
        model.set_weights(initial_weights); best_epoch = 0
    for before, after in zip(frozen_weights, backbone.get_weights()):
        np.testing.assert_array_equal(before, after)
    set_proba("M10", "tune", predict_loader(model, loaders["tune"]))
    set_proba("M10", "holdout", predict_loader(model, _hybrid_inputs("holdout", INFER_BATCH)))
    register("M10", family="hybrid", input="table + listening",
             n_channels=8, n_parameters=int(model.count_params()),
             best_params=bp, trials=counts, fit_seconds=time.perf_counter() - t0,
             best_epoch=best_epoch, history=hist.history, estimator=model,
             sequence_correction_kept=kept, batch_size=bp["batch_size"], pool=bp["pool"],
             mlp_params=REGISTRY["M9"]["mlp_params"])
    print(f"M10 correction kept={kept} best_epoch={best_epoch}")

run_M10()

## 10. Thresholds, development refit and the single holdout evaluation

Thresholds come from maximum churn F1 on **tune** (February, deterministic tie rule). The
primary and sequential winners are chosen by tune log-loss and written to a manifest **before**
the March holdout is touched. Each trainable family is refit on all of February (preprocessors
refit on February only), and the models are evaluated once on March. Thresholds are not
re-searched on March; how a February threshold behaves one month later is part of the honest
report.


In [ ]:
# ---- Thresholds and winner selection on tune (before holdout) ------------
MODEL_IDS = ["M5", "M7", "M8", "M10"]
THRESHOLDS = {}
for mid in MODEL_IDS:
    THRESHOLDS[mid] = select_threshold(y_tune, PROBA[mid]["tune"])

tune_losses = {mid: log_loss(y_tune, PROBA[mid]["tune"], labels=[0, 1]) for mid in MODEL_IDS}
primary_winner = min(MODEL_IDS, key=lambda m: tune_losses[m])
sequential_ids = ["M5", "M7", "M8"] + (
    ["M10"] if REGISTRY["M10"].get("sequence_correction_kept") else [])
sequential_winner = min(sequential_ids, key=lambda m: tune_losses[m])

SELECTION = {
    "contract_hash": CONTRACT_HASH,
    "tune_log_loss": {m: round(float(v), 6) for m, v in tune_losses.items()},
    "thresholds": {m: round(float(t), 4) for m, t in THRESHOLDS.items()},
    "primary_winner": primary_winner,
    "sequential_winner": sequential_winner,
    "m10_correction_kept": bool(REGISTRY["M10"].get("sequence_correction_kept")),
}
with (OUT_DIR / "selection_manifest.json").open("w") as f:
    json.dump(SELECTION, f, indent=2)
print("primary winner (tune):", primary_winner,
      "| sequential winner:", sequential_winner)
print("thresholds:", {m: round(t, 3) for m, t in THRESHOLDS.items()})


In [ ]:
# ---- Development refit: retrain each family on all of February ------------
# Preprocessors use the development phase; no holdout (March) early stop.
DEV = "development"
y_dev = y_all[DEV_ROWS]
hold_rows = rows_of("holdout")
y_hold = y_all[hold_rows]

HOLD_PROBA = {}   # model_id -> holdout probability from the development-refit model


In [ ]:
# Sequence models refit on all development (preprocessors on development phase).
def _refit_sequence(model_id):
    bp = REGISTRY[model_id]["best_params"]
    second = bp.get("second_units") if bp["n_layers"] == 2 else None
    input_shape, _ = _seq_input_shape(model_id)
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)
    model = build_sequence_model(input_shape, float(y_dev.mean()), bp["units"], second,
                                 bp["dropout"], bp["lr"], bp["pool"])
    # fit on development, early stop on a small internal stop drawn from development tune part
    fit_loader = SequenceLoader(DEV_ROWS, SOURCE_OF[model_id], DEV, bp["batch_size"],
                                labels=y_dev, shuffle=True)
    stop_loader = SequenceLoader(rows_of("stop"), SOURCE_OF[model_id], DEV,
                                 bp["batch_size"], labels=y_of("stop"))
    model.fit(fit_loader, validation_data=stop_loader, epochs=DL_MAX_EPOCHS,
              verbose=TRAIN_VERBOSE, callbacks=dl_callbacks())
    hold_loader = SequenceLoader(hold_rows, SOURCE_OF[model_id], DEV, INFER_BATCH)
    HOLD_PROBA[model_id] = predict_loader(model, hold_loader)
    REGISTRY[model_id]["dev_estimator"] = model

for _mid in ("M5", "M7", "M8"):
    _refit_sequence(_mid)

In [ ]:
# M9 refit on development; M10 refit using the development M9 frozen.
_m9p = REGISTRY["M9"]["mlp_params"]
_reuse_m9_dev = ("_m9" in globals() and REGISTRY["M9"].get("dev_estimator") is _m9
                 and "M9" in HOLD_PROBA and "T_dev" in globals()
                 and "T_stop_dev" in globals()
                 and T_dev.shape[0] == len(DEV_ROWS)
                 and T_stop_dev.shape[0] == len(rows_of("stop")))
if _reuse_m9_dev:
    print("Reusing the completed development M9 refit and design matrices.", flush=True)
else:
    print("Preparing development tabular matrices for M9.", flush=True)
    T_dev = tabular_batch(DEV_ROWS, DEV)
    T_stop_dev = tabular_batch(rows_of("stop"), DEV)
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)
    _m9 = build_tabular_model(T_dev.shape[1], float(y_dev.mean()), _m9p["units_1"],
                              _m9p["units_2"], _m9p["dropout"], _m9p["lr"])
    _m9.fit(T_dev, y_dev, validation_data=(T_stop_dev, y_of("stop")),
            epochs=DL_MAX_EPOCHS, batch_size=_m9p["batch_size"],
            verbose=TRAIN_VERBOSE, callbacks=dl_callbacks())
    T_hold_dev = tabular_batch(hold_rows, DEV)
    HOLD_PROBA["M9"] = np.clip(_m9.predict(T_hold_dev, batch_size=INFER_BATCH,
                                            verbose=0).ravel(), EPS, 1 - EPS)
    REGISTRY["M9"]["dev_estimator"] = _m9

# M10 development refit: freeze dev M9, train correction with the selected budget.
_bp10 = REGISTRY["M10"]["best_params"]
_second10 = _bp10.get("second_units") if _bp10["n_layers"] == 2 else None
_backbone_dev = _frozen_backbone(_m9)
_frozen_dev = _backbone_dev.get_weights()
_seq_shape, _ = _seq_input_shape("M5")
tf.keras.utils.set_random_seed(SEED)
_m10 = build_hybrid_model(_backbone_dev, T_dev.shape[1], _seq_shape, _bp10["units"],
                          _second10, _bp10["dropout"], _bp10["lr"], _bp10["pool"])
_init10 = _m10.get_weights()
def _hybrid_dev_loader(rows, labels, shuffle=False, batch_size=None):
    size = _bp10["batch_size"] if batch_size is None else batch_size
    return SequenceLoader(rows, "listening", DEV, size,
                          labels=labels, hybrid=True, shuffle=shuffle)
_mlp_stop_dev = np.clip(_m9.predict(T_stop_dev, batch_size=INFER_BATCH, verbose=0).ravel(), EPS, 1 - EPS)
_stop_hybrid_dev = _hybrid_dev_loader(rows_of("stop"), y_of("stop"), batch_size=INFER_BATCH)
np.testing.assert_array_equal(_stop_hybrid_dev.rows, rows_of("stop"))
for weight in _m10.get_layer("sequence_correction").get_weights():
    assert np.count_nonzero(weight) == 0
_initial_hybrid_stop_dev = predict_loader(_m10, _stop_hybrid_dev)
print("M10 development initial MLP/hybrid max difference:",
      float(np.max(np.abs(_initial_hybrid_stop_dev - _mlp_stop_dev))))
_init_loss10 = log_loss(y_of("stop"), _initial_hybrid_stop_dev, labels=[0, 1])
_m10.fit(_hybrid_dev_loader(DEV_ROWS, y_dev, shuffle=True),
         validation_data=_hybrid_dev_loader(rows_of("stop"), y_of("stop")),
         epochs=DL_MAX_EPOCHS, verbose=TRAIN_VERBOSE, callbacks=dl_callbacks())
_trained_loss10 = log_loss(y_of("stop"), predict_loader(_m10, _stop_hybrid_dev),
                           labels=[0, 1])
# Keep the development-phase policy (do not decide using holdout).
if not (_trained_loss10 < _init_loss10 - 1e-5):
    _m10.set_weights(_init10)
    REGISTRY["M10"]["sequence_correction_kept_dev"] = False
else:
    REGISTRY["M10"]["sequence_correction_kept_dev"] = True
for b, a in zip(_frozen_dev, _backbone_dev.get_weights()):
    np.testing.assert_array_equal(b, a)
HOLD_PROBA["M10"] = predict_loader(
    _m10, _hybrid_dev_loader(hold_rows, None, batch_size=INFER_BATCH))
REGISTRY["M10"]["dev_estimator"] = _m10
print("M10 dev correction kept:", REGISTRY["M10"]["sequence_correction_kept_dev"])

## 11. Metrics tables, confusion matrices and curves

Holdout (March) metrics for all neural models at their tune threshold, plus a complementary table at
threshold 0.5. Confusion matrices (counts + row-normalised) per model and as a paginated
gallery. ROC/PR curves, reliability diagrams, deep-learning learning curves and Optuna
histories. Order the display table by holdout log-loss for inspection, distinct from the
winner fixed earlier on tune.


In [ ]:
# ---- Holdout metrics tables (tune threshold and 0.5) ----------------------
FIG_DIR = OUT_DIR / "figures"
(FIG_DIR / "confusion").mkdir(parents=True, exist_ok=True)
(FIG_DIR / "curves").mkdir(parents=True, exist_ok=True)
REPORT_DIR = OUT_DIR / "metrics" / "classification_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

rows_main, rows_half = [], []
for mid in MODEL_IDS:
    rec = REGISTRY[mid]
    row = evaluate(y_hold, PROBA[mid]["holdout"], mid, THRESHOLDS[mid])
    row.update({
        "family": rec["family"], "input": rec["input"],
        "n_channels": rec.get("n_channels", "N/A"),
        "n_parameters": rec.get("n_parameters", "N/A"),
        "trials_complete": rec["trials"]["complete"],
        "trials_pruned": rec["trials"]["pruned"],
        "trials_failed": rec["trials"]["failed"],
        "fit_seconds": round(float(rec.get("fit_seconds", 0.0)), 2),
        "best_epoch": rec.get("best_epoch", "N/A"),
        "correction_kept": rec.get("sequence_correction_kept_dev",
                                   rec.get("sequence_correction_kept", "N/A")),
        "best_params": json.dumps(rec.get("best_params", "N/A")),
    })
    rows_main.append(row)
    rows_half.append(metrics_at_half(y_hold, PROBA[mid]["holdout"], mid))
    save_classification_report(y_hold, PROBA[mid]["holdout"], THRESHOLDS[mid], mid, REPORT_DIR)

holdout_metrics = pd.DataFrame(rows_main).sort_values("log_loss").reset_index(drop=True)
holdout_metrics_at_05 = pd.DataFrame(rows_half).sort_values("log_loss").reset_index(drop=True)
holdout_metrics.to_csv(OUT_DIR / "metrics" / "holdout_metrics.csv", index=False)
holdout_metrics_at_05.to_csv(OUT_DIR / "metrics" / "holdout_metrics_at_05.csv", index=False)

_display_cols = ["model_id", "family", "log_loss", "roc_auc", "average_precision",
                 "brier", "accuracy", "balanced_accuracy", "precision_churn",
                 "recall_churn", "specificity", "f1_churn", "f1_macro", "mcc", "threshold"]
# Tune (February) vs holdout (March) log-loss: the gap is the month-over-month drift.
holdout_metrics["tune_log_loss"] = holdout_metrics["model_id"].map(SELECTION["tune_log_loss"])
holdout_metrics["tune_churn_rate"] = float(y_tune.mean())
_display_cols.insert(3, "tune_log_loss")
print("Holdout = March cohort; tune = February. Prevalence tune/holdout:",
      round(float(y_tune.mean()), 4), "/", round(float(y_hold.mean()), 4))
print("Holdout metrics ordered by log-loss (winner fixed on tune:", primary_winner, ")")
holdout_metrics[_display_cols].round(5)

In [ ]:
# ---- Confusion matrices: per-model images + paginated gallery -------------
hold_entries = {mid: (y_hold, PROBA[mid]["holdout"]) for mid in MODEL_IDS}
for mid in MODEL_IDS:
    fig = save_confusion(y_hold, PROBA[mid]["holdout"], THRESHOLDS[mid], mid,
                         FIG_DIR / "confusion", tag="holdout")
    plt.close(fig)
_gallery = gallery_confusion(hold_entries, THRESHOLDS, FIG_DIR / "confusion",
                             tag="holdout", per_page=3)
for f in _gallery:
    plt.show()
print("saved per-model confusion images (PNG+SVG) and gallery pages")

In [ ]:
# ---- ROC and precision-recall curves --------------------------------------
fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(14, 6))
for mid in MODEL_IDS:
    p = PROBA[mid]["holdout"]
    fpr, tpr, _ = roc_curve(y_hold, p)
    ax_roc.plot(fpr, tpr, label=f"{mid} (AUC={roc_auc_score(y_hold, p):.3f})")
    prec, rec, _ = precision_recall_curve(y_hold, p)
    ax_pr.plot(rec, prec, label=f"{mid} (AP={average_precision_score(y_hold, p):.3f})")
ax_roc.plot([0, 1], [0, 1], "k--", alpha=0.5, label="chance")
ax_roc.set(title="ROC (holdout)", xlabel="FPR", ylabel="TPR"); ax_roc.legend(fontsize=7)
ax_pr.axhline(float(y_hold.mean()), color="k", ls="--", alpha=0.5, label="prevalence")
ax_pr.set(title="Precision-Recall (holdout)", xlabel="Recall", ylabel="Precision")
ax_pr.legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIG_DIR / "curves" / "roc_pr.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# ---- Reliability diagrams and probability histograms (diagnostics only) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for mid in MODEL_IDS:
    p = PROBA[mid]["holdout"]
    frac_pos, mean_pred = calibration_curve(y_hold, p, n_bins=10, strategy="quantile")
    axes[0].plot(mean_pred, frac_pos, marker="o", label=mid, alpha=0.7)
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[0].set(title="Reliability (holdout, diagnostic only)",
            xlabel="Mean predicted probability", ylabel="Observed churn rate")
axes[0].legend(fontsize=7)
best_p = PROBA[primary_winner]["holdout"]
axes[1].hist(best_p[y_hold == 0], bins=40, alpha=0.6, label="Permanece (0)", density=True)
axes[1].hist(best_p[y_hold == 1], bins=40, alpha=0.6, label="Churn (1)", density=True)
axes[1].set(title=f"Probability histogram — {primary_winner}", xlabel="probability")
axes[1].legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "curves" / "calibration_hist.png", dpi=160, bbox_inches="tight")
plt.show()
print("NOTE: reliability curves are diagnostics; no probability calibration was applied.")

In [ ]:
# ---- Learning curves (DL fit/stop) and Optuna histories -------------------
dl_models = [m for m in MODEL_IDS if "history" in REGISTRY[m]]
if dl_models:
    fig, axes = plt.subplots(1, len(dl_models), figsize=(4 * len(dl_models), 4), squeeze=False)
    for ax, mid in zip(axes[0], dl_models):
        h = REGISTRY[mid]["history"]
        ep = np.arange(1, len(h["loss"]) + 1)
        ax.plot(ep, h["loss"], label="fit")
        ax.plot(ep, h["val_loss"], label="stop")
        ax.set(title=f"{mid} learning curve", xlabel="epoch", ylabel="log-loss")
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "curves" / "learning_curves.png", dpi=160, bbox_inches="tight")
    plt.show()

# Optuna optimisation history per family with enough completed trials.
for mid in ["M5", "M7", "M8", "M10"]:
    try:
        study = get_study(mid)
    except Exception:
        continue
    complete = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    if len(complete) >= 3:
        ax = optuna.visualization.matplotlib.plot_optimization_history(study)
        ax.set_title(f"Optuna history — {mid}")
        plt.show()
    else:
        print(f"{mid}: {len(complete)} completed trials — history plot skipped (need >= 3).")


## 12. Final refit on February + March and reloadable scoring

Decisions are frozen before holdout. Now retrain M5, M7, M8 and M9 on both labelled months
(February + March) with the selected params/budgets; M10 on the new frozen M9. April is
predicted with the most recent and largest training set. Preprocessors are refit on those rows
only here; epochs keep the early-stopped count from February. Test has no labels: it never
produces evaluation metrics. We score test in blocks, restore the original msno order, and save
one submission per model plus `submission.csv` (primary winner) and `submission_sequential.csv`
(best sequential).

A load function (`load_predictor`) rebuilds each predictor from its manifest without notebook
globals. `score_model(model_id, ...)` validates inputs, joins by unique msno and uses the
saved threshold; it never silently falls back to another model.


In [ ]:
# Final-phase scalers (fit on February + March) and the final design matrices.
TRAIN_ROWS = PHASE_ROWS["train"]
y_train_all = y_all[TRAIN_ROWS]
FINAL_RATE = float(y_train_all.mean())

MODELS_DIR = OUT_DIR / "models" / "final"
PRE_DIR = OUT_DIR / "preprocessing" / "final"
PRED_DIR = OUT_DIR / "predictions"
SUB_DIR = OUT_DIR / "submissions"
for d in (MODELS_DIR, PRE_DIR, PRED_DIR, SUB_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Persist the final preprocessors (fit on February + March).
_listen_f, _tx_f, _tab_f = phase_scalers("train")
for fname, state in [("sequence_scaler.json", _listen_f),
                     ("transaction_scaler.json", _tx_f),
                     ("tabular_preprocess.json", _tab_f)]:
    with (PRE_DIR / fname).open("w") as f:
        json.dump(state, f, indent=2)

FINAL_MODELS = {}   # model_id -> fitted final estimator (trees/sk) or keras path
TEST_PROBA = {}
test_all_rows = np.arange(len(test_df) if SMOKE_TEST_ROWS is None else SMOKE_TEST_ROWS)
scored_test = test_df.iloc[test_all_rows]

In [ ]:
# ---- Final sequence models (M5, M7, M8) on February + March --------------
def _final_sequence(model_id):
    bp = REGISTRY[model_id]["best_params"]
    second = bp.get("second_units") if bp["n_layers"] == 2 else None
    input_shape, _ = _seq_input_shape(model_id)
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)
    model = build_sequence_model(input_shape, FINAL_RATE, bp["units"], second,
                                 bp["dropout"], bp["lr"], bp["pool"])
    epochs = max(1, REGISTRY[model_id].get("best_epoch", DL_MAX_EPOCHS) or DL_MAX_EPOCHS)
    train_loader = SequenceLoader(TRAIN_ROWS, SOURCE_OF[model_id], "train",
                                  bp["batch_size"], labels=y_train_all, shuffle=True)
    model.fit(train_loader, epochs=epochs, verbose=TRAIN_VERBOSE)
    test_loader = SequenceLoader(test_all_rows, SOURCE_OF[model_id], "train",
                                 INFER_BATCH, dataset="test")
    TEST_PROBA[model_id] = predict_loader(model, test_loader)
    path = MODELS_DIR / f"{model_id}_sequence.keras"
    model.save(str(path))
    FINAL_MODELS[model_id] = path
    return model

for _mid in ("M5", "M7", "M8"):
    _final_sequence(_mid)
print("final sequence models trained on February + March")

In [ ]:
# ---- Final M9 (February + March) and M10 (frozen final M9) ---------------
_m9p = REGISTRY["M9"]["mlp_params"]
T_train = tabular_batch(TRAIN_ROWS, "train")
tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)
m9_final = build_tabular_model(T_train.shape[1], FINAL_RATE, _m9p["units_1"],
                               _m9p["units_2"], _m9p["dropout"], _m9p["lr"])
_m9_epochs = max(1, REGISTRY["M9"].get("best_epoch", DL_MAX_EPOCHS) or DL_MAX_EPOCHS)
m9_final.fit(T_train, y_train_all, epochs=_m9_epochs, batch_size=_m9p["batch_size"], verbose=TRAIN_VERBOSE)
T_test = tabular_batch(test_all_rows, "train", dataset="test")
TEST_PROBA["M9"] = np.clip(m9_final.predict(T_test, batch_size=INFER_BATCH, verbose=0).ravel(),
                           EPS, 1 - EPS)
m9_final.save(str(MODELS_DIR / "M9_mlp.keras"))
FINAL_MODELS["M9"] = MODELS_DIR / "M9_mlp.keras"

# M10 final: freeze final M9, train correction with the development policy and budget.
_bp10 = REGISTRY["M10"]["best_params"]
_second10 = _bp10.get("second_units") if _bp10["n_layers"] == 2 else None
backbone_final = _frozen_backbone(m9_final)
_frozen_final = backbone_final.get_weights()
_seq_shape, _ = _seq_input_shape("M5")
tf.keras.utils.set_random_seed(SEED)
m10_final = build_hybrid_model(backbone_final, T_train.shape[1], _seq_shape, _bp10["units"],
                               _second10, _bp10["dropout"], _bp10["lr"], _bp10["pool"])
_init10f = m10_final.get_weights()
_m10_epochs = max(1, REGISTRY["M10"].get("best_epoch", DL_MAX_EPOCHS) or DL_MAX_EPOCHS)
# Honour the policy chosen in development: only train the correction if it was kept.
if REGISTRY["M10"].get("sequence_correction_kept_dev", False):
    train_hybrid = SequenceLoader(TRAIN_ROWS, "listening", "train", _bp10["batch_size"],
                                  labels=y_train_all, hybrid=True, shuffle=True)
    m10_final.fit(train_hybrid, epochs=_m10_epochs, verbose=TRAIN_VERBOSE)
else:
    print("M10: development policy restored zero correction -> M10 equals the MLP.")
for b, a in zip(_frozen_final, backbone_final.get_weights()):
    np.testing.assert_array_equal(b, a)
test_hybrid = SequenceLoader(test_all_rows, "listening", "train", INFER_BATCH,
                             hybrid=True, dataset="test")
TEST_PROBA["M10"] = predict_loader(m10_final, test_hybrid)
m10_final.save(str(MODELS_DIR / "M10_hybrid.keras"))
FINAL_MODELS["M10"] = MODELS_DIR / "M10_hybrid.keras"
print("final M9/M10 trained on February + March")

In [ ]:
# ---- Final predictions validation and submissions -------------------------
# Validate ranges/length/finiteness and restore original msno order (already original).
for mid in MODEL_IDS:
    p = np.asarray(TEST_PROBA[mid], dtype=np.float64)
    assert p.shape == (len(scored_test),), f"{mid} wrong length"
    assert np.isfinite(p).all() and (p >= 0).all() and (p <= 1).all(), f"{mid} invalid probs"
    pd.DataFrame({"msno": scored_test["msno"].to_numpy(), "is_churn": p}).to_parquet(
        PRED_DIR / f"test_{mid}.parquet", index=False)

# Submissions: one per model, the primary winner and the best sequential winner.
for mid in MODEL_IDS:
    pd.DataFrame({"msno": scored_test["msno"].to_numpy(), "is_churn": TEST_PROBA[mid]}).to_csv(
        SUB_DIR / f"submission_{mid}.csv", index=False)
pd.DataFrame({"msno": scored_test["msno"].to_numpy(), "is_churn": TEST_PROBA[primary_winner]}).to_csv(
    SUB_DIR / "submission.csv", index=False)
pd.DataFrame({"msno": scored_test["msno"].to_numpy(), "is_churn": TEST_PROBA[sequential_winner]}).to_csv(
    SUB_DIR / "submission_sequential.csv", index=False)
print("submissions written. primary:", primary_winner, "| sequential:", sequential_winner)


In [ ]:
# ---- Per-model manifests and a reloadable predictor interface -------------
def _manifest(mid):
    rec = REGISTRY[mid]
    man = {
        "model_id": mid, "family": rec["family"], "input": rec["input"],
        "dataset_version": SEQ_META["dataset_version"], "contract_hash": CONTRACT_HASH,
        "n_channels": rec.get("n_channels", "N/A"),
        "channels": rec.get("channels", "N/A"),
        "best_params": rec.get("best_params", "N/A"),
        "threshold": THRESHOLDS[mid], "fit_rate": FINAL_RATE,
        "libs": ENV, "seed": SEED,
        "preprocessing": {"sequence": "preprocessing/final/sequence_scaler.json",
                          "transaction": "preprocessing/final/transaction_scaler.json",
                          "tabular": "preprocessing/final/tabular_preprocess.json"},
    }
    if mid in ("M5", "M7", "M8"):
        man["source"] = SOURCE_OF[mid]
        man["model_file"] = f"models/final/{mid}_sequence.keras"
    if mid == "M10":
        man["model_file"] = "models/final/M10_hybrid.keras"
        man["correction_kept"] = REGISTRY["M10"].get("sequence_correction_kept_dev", False)
    return man

MANIFEST_DIR = OUT_DIR / "models" / "manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
MANIFESTS = {mid: _manifest(mid) for mid in MODEL_IDS}
for mid, man in MANIFESTS.items():
    with (MANIFEST_DIR / f"{mid}.json").open("w") as f:
        json.dump(man, f, indent=2, default=str)
holdout_metrics.to_csv(OUT_DIR / "metrics" / "holdout_metrics.csv", index=False)
with (OUT_DIR / "run_config.json").open("w") as f:
    json.dump({"env": ENV, "smoke": SMOKE_TEST, "trials": TRIALS,
               "dl_max_epochs": DL_MAX_EPOCHS, "contract": CONTRACT,
               "selection": SELECTION}, f, indent=2, default=str)
print("saved manifests, run_config.json and metrics")


In [ ]:
# ---- Reloadable scoring by model_id (no notebook globals needed) ----------
def _load_scalers(man):
    base = OUT_DIR
    listen = json.loads((base / man["preprocessing"]["sequence"]).read_text())
    tx = json.loads((base / man["preprocessing"]["transaction"]).read_text())
    tab = json.loads((base / man["preprocessing"]["tabular"]).read_text())
    return listen, tx, tab


def load_predictor(model_id):
    """Rebuild a predictor from its manifest + artifacts, independent of run state."""
    man = json.loads((MANIFEST_DIR / f"{model_id}.json").read_text())
    family = man["family"]
    listen, tx, tab = _load_scalers(man)
    keras_model = tf.keras.models.load_model(str(OUT_DIR / man["model_file"]))

    def predict_keras(feats, seq=None):
        if family == "mlp":
            return np.clip(keras_model.predict(transform_tabular(feats, tab),
                                               batch_size=INFER_BATCH, verbose=0).ravel(),
                           EPS, 1 - EPS)
        if seq is None:
            raise ValueError(f"{model_id} requires the sequence tensor; none provided.")
        raw = np.asarray(seq[:, -N_DAYS:, :], dtype=np.float32)
        if man.get("source") == "listening":
            x = transform_sequence(raw[:, :, :8], listen)
        elif man.get("source") == "transaction":
            x = transform_combined(raw, listen, tx, TX_SPECS)[:, :, 8:]
        elif man.get("source") == "combined":
            x = transform_combined(raw, listen, tx, TX_SPECS)
        else:  # hybrid uses listening + tabular
            x = {"tabular": transform_tabular(feats, tab),
                 "sequence": transform_sequence(raw[:, :, :8], listen)}
        return np.clip(keras_model.predict(x, batch_size=INFER_BATCH, verbose=0).ravel(),
                       EPS, 1 - EPS)
    return predict_keras, man


def score_model(model_id, features_df, seq=None):
    """Return msno + probability for one model id, by its manifest. Errors are explicit."""
    predictor, man = load_predictor(model_id)
    proba = predictor(features_df, seq)
    proba = np.clip(np.asarray(proba, dtype=np.float64), EPS, 1 - EPS)
    return pd.DataFrame({"msno": features_df["msno"], "is_churn": proba})


In [ ]:
# ---- Reload check: final scoring matches in-memory inference (tolerance) --
_check_id = primary_winner
_rows_check = test_all_rows[:256]
_feats_check = test_df.iloc[_rows_check]
_seq_check = None
if _check_id in ("M5", "M7", "M8", "M10"):
    _fam = "lstm" if _check_id in ("M7", "M8") else "hybrid"
    _seq_check = np.asarray(SEQ[(_fam, "test")][_rows_check], dtype=np.float32)
_reloaded = score_model(_check_id, _feats_check, _seq_check)["is_churn"].to_numpy()
_in_mem = np.asarray(TEST_PROBA[_check_id])[_rows_check]
print(f"reload check for {_check_id}: max abs diff =",
      float(np.max(np.abs(_reloaded - _in_mem))))
assert np.allclose(_reloaded, _in_mem, atol=1e-4), "reloaded scoring diverged"
print("reloadable scoring verified within tolerance")


## 13. Notes

**What this notebook does.** Three sequence models (M5 listening, M7 transaction, M8
combined) and one hybrid (M10 frozen MLP + listening LSTM) share one split
(`fit`/`stop`/`tune`/`holdout`). M9 is trained only as M10's backbone. Each trainable family
has its own Optuna study persisted in SQLite (resumable by total budget, rejecting a changed
data/split contract). Metrics, confusion matrices (counts + row-normalised), ROC/PR,
reliability, learning curves and Optuna histories are produced for every compared model.
Winners and thresholds are fixed on `tune` (February) and written to `selection_manifest.json`
**before** the March holdout is read, then families are refit on all of February and evaluated
once on March, and finally refit on February + March for April test inference. Reloadable scoring (`load_predictor` /
`score_model`) rebuilds each predictor from its manifest without notebook globals.

**Honest scope of results.**
- `SMOKE_TEST=True` uses a small user sample, few trials and few epochs. Its numbers are
  **not** FULL results and go to `*_smoke/`. Set `SMOKE_TEST=False` and run on a GPU for the
  full comparison.
- The holdout is cross-month: models trained on February (sequences ending 2017-01-31) are
  scored on March (sequences ending 2017-02-28). February labels come from the official v1
  `train.csv`, March labels from `train_v2.csv`; prevalence differs (≈ 6.4 % vs ≈ 9.0 %).
- Thresholds are fixed on February and not re-tuned on March.
- Reliability curves are diagnostics; no probability calibration was applied.
- The hybrid M10 restores its zero correction if it does not beat `stop`; in that case it is
  "equivalent to the MLP" and is excluded from the sequential-winner choice.

**Artifacts** are written under `OUT_DIR` (`run_config.json`, `split_indices.npz`,
`split_summary.csv`, `selection_manifest.json`, `optuna/`, `preprocessing/`, `models/`,
`metrics/`, `predictions/`, `figures/`, `submissions/`). No submission is uploaded to
Kaggle automatically.

The non-neural models (logistic, RandomForest, XGBoost) live in the w1 tabular
notebook.
